[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheffieldML/notebook/blob/main/GPc/olympic_marathon.ipynb)

# Olympic marathon with `gpc` + `pods`

Olympics-style 1-D regression on men’s marathon winning times (CIP-001D Phase E + CIP-0031). Modelling uses core `gpc` only; the dataset comes from external [`pods`](https://pypi.org/project/pods/) — the same lab data path many GPy courses use.

Years sit on an \(O(2000)\) scale with multi-decade gaps. A default `rbf.lengthscale=1` is nonsense in that unit system: set the lengthscale in **years** (here 30) before optimising, for both full and sparse fits.

This notebook also teaches the CIP-0031 contract: `predict` returns the noisy predictive \(y_*\) (latent + `likelihood.variance`); `predict_noiseless` returns \(f_*\). `plot_1d` defaults to \(y_*\); pass `noiseless=True` for \(f_*\).

```bash
pip install pods matplotlib   # or: pip install 'gpc[examples]'
```


In [ ]:
import importlib
import re
import subprocess
import sys


def _gpc_ver_tuple(v: str):
    """Parse ``0.1.0a10`` / ``0.1.0`` into a comparable tuple (final > any alpha)."""
    m = re.fullmatch(r"(\d+)\.(\d+)\.(\d+)(?:a(\d+))?", str(v))
    if not m:
        return (0, 0, 0, 0)
    maj, mir, pat, alpha = m.groups()
    pre = int(alpha) if alpha is not None else 10**9
    return (int(maj), int(mir), int(pat), pre)


_MIN = "0.1.0a10"
_need = _gpc_ver_tuple(_MIN)


def _ensure_gpc():
    try:
        import gpc as _gpc
        if _gpc_ver_tuple(getattr(_gpc, "__version__", "0")) >= _need:
            return _gpc
    except ImportError:
        pass
    # Prefer TestPyPI for pre-release alphas; real PyPI for dependencies (CIP-002C).
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "--no-cache-dir",
            "--index-url",
            "https://test.pypi.org/simple/",
            "--extra-index-url",
            "https://pypi.org/simple/",
            f"gpc>={_MIN}",
        ]
    )
    if "gpc" in sys.modules:
        importlib.reload(sys.modules["gpc"])
    import gpc as _gpc
    if _gpc_ver_tuple(getattr(_gpc, "__version__", "0")) < _need:
        raise RuntimeError(
            f"need gpc>={_MIN}, got {getattr(_gpc, '__version__', '?')}"
        )
    return _gpc


gpc = _ensure_gpc()


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import numpy as np
import gpc

try:
    from matplotlib import pyplot as plt
    from gpc.plot import plot_1d
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])
    from matplotlib import pyplot as plt
    from gpc.plot import plot_1d

try:
    import pods
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pods"])
    import pods


## Full GP on marathon times

Load `pods.datasets.olympic_marathon_men()`, fit an RBF GP with a year-scale lengthscale, and print paths / `objective()` (**−log-likelihood**).


In [ ]:
import numpy as np
import gpc

try:
    import pods
except ImportError as exc:
    raise SystemExit("pip install pods  (or gpc[examples])") from exc

data = pods.datasets.olympic_marathon_men()
X = np.asarray(data["X"], dtype=float)
Y = np.asarray(data["Y"], dtype=float)
if X.ndim == 1:
    X = X[:, None]
if Y.ndim == 1:
    Y = Y[:, None]

m = gpc.GPRegression(X, Y)
m.set_by_path("rbf.lengthscale", 30.0)
m.optimize(iters=100, messages=True)
print("objective:", m.objective())
print({p: m.get_by_path(p) for p in m.param_paths()})


## `predict` vs `predict_noiseless`

At the training inputs, \(\mathrm{var}(y_*)-\mathrm{var}(f_*)\) should match `likelihood.variance` (CIP-0031). The left panel is the observation-scale band; the right panel is the latent function band.


In [ ]:
from gpc.plot import plot_1d
import matplotlib.pyplot as plt

mu_y, var_y = m.predict(X)
mu_f, var_f = m.predict_noiseless(X)
sigma2 = m.get_by_path("likelihood.variance")
print("likelihood.variance:", sigma2)
print("mean(var_y - var_f) at train X:", float(np.mean(var_y - var_f)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
plot_1d(m, X, Y, ax=axes[0], label="y_*", noiseless=False)
axes[0].set_title("predict (y_*)")
axes[0].set_xlabel("year")
axes[0].set_ylabel("time (hours)")
plot_1d(m, X, Y, ax=axes[1], label="f_*", noiseless=True)
axes[1].set_title("predict_noiseless (f_*)")
axes[1].set_xlabel("year")
plt.tight_layout()
plt.show()

## Sparse DTCVAR variant

Place inducing inputs on a year grid and compare the sparse bound to the full GP.

**Init.** Years are \(O(2000)\) with gaps of decades. Default `rbf.lengthscale=1` makes neighbouring inducing points look almost identical, so **DTC** can hit a non-positive-definite `K_uu` and **DTCVAR** can “succeed” into a flat, high-noise fit. Set the lengthscale in years (same as the FTC cell). `rbf.variance=1` is fine once the lengthscale matches the data scale — bumping variance alone does not fix the conditioning.

This is the same compression story as [sparse_gp_regression](./sparse_gp_regression.ipynb), on real lab data.


In [ ]:
Z = np.linspace(float(X.min()), float(X.max()), 8)[:, None]
ms = gpc.SparseGPRegression(X, Y, Z)  # approx='dtcvar' default
ms.set_by_path("rbf.lengthscale", 30.0)  # year-scale; variance=1 is OK
ms.optimize(iters=80, messages=True)
print("sparse objective:", ms.objective(), "approx:", ms.approx)
print({p: ms.get_by_path(p) for p in ms.param_paths() if "inducing" not in p})
mu_y, var_y = ms.predict(X)
mu_f, var_f = ms.predict_noiseless(X)
print("sparse mean(var_y - var_f):", float(np.mean(var_y - var_f)),
      "likelihood.variance:", ms.get_by_path("likelihood.variance"))
plot_1d(ms, X, Y, label="dtcvar y_*")
plt.show()

# Optional: DTC with the same init (default ls=1 often raises NPD on this grid).
# md = gpc.SparseGPRegression(X, Y, Z, approx="dtc")
# md.set_by_path("rbf.lengthscale", 30.0)
# md.optimize(iters=80, messages=True)